In [10]:
"""
Download one year of ACN-Data sessions for one site (no time series).
Setup:  pip install requests pandas
        export ACN_API_TOKEN="your-token"     (in Codespaces: add it as a secret)
Run:    python download_acn.py
"""

import os
import time
from email.utils import format_datetime
from pathlib import Path
from urllib.parse import urljoin
from dotenv import load_dotenv

import pandas as pd
import requests

# ---------- settings ----------
SITE = "jpl"            # "caltech", "jpl" or "office001"
START = None            # e.g. "2019-01-01". None = start at the site's first session
MONTHS = 12             # length of "year 1"
OUT = Path("data")      # keep this folder in .gitignore
BASE = "https://ev.caltech.edu/api/v1/"
load_dotenv()  # reads .env from the project root
TOKEN = os.environ["API_TOKEN"]
# ------------------------------

OUT.mkdir(exist_ok=True)
session = requests.Session()
session.auth = (TOKEN, "")          # token as username, blank password


def get_json(url, params=None, retries=3):
    for attempt in range(retries):
        r = session.get(url, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        print(f"  HTTP {r.status_code}, retrying ({attempt + 1}/{retries})")
        time.sleep(2 * (attempt + 1))
    r.raise_for_status()


def items_of(page):
    # The page structure shown in the docs has "_links" and "_meta";
    # the list of sessions is expected under "_items".
    return page.get("_items") or page.get("items") or []


def rfc1123(ts):
    return format_datetime(ts.to_pydatetime(), usegmt=True)


def fetch_range(start, end):
    """All sessions with start <= connectionTime < end, following 'next' links."""
    where = f'connectionTime>="{rfc1123(start)}" and connectionTime<"{rfc1123(end)}"'
    page = get_json(urljoin(BASE, f"sessions/{SITE}"), params={"where": where})
    rows = []
    while True:
        rows.extend(items_of(page))
        nxt = page.get("_links", {}).get("next")
        if not nxt:
            break
        page = get_json(urljoin(BASE, nxt["href"]))
        time.sleep(0.2)             # be gentle with their server
    return rows


def first_session_time():
    # Per the docs, "-connectionTime" sorts ascending, so the oldest sessions come first.
    page = get_json(urljoin(BASE, f"sessions/{SITE}"), params={"sort": "-connectionTime"})
    times = pd.to_datetime([s["connectionTime"] for s in items_of(page)], utc=True)
    return times.min()


def flatten(s):
    row = {k: s.get(k) for k in [
        "sessionID", "siteID", "clusterID", "stationID", "spaceID", "userID",
        "connectionTime", "disconnectTime", "doneChargingTime",
        "kWhDelivered", "timezone"]}
    inputs = s.get("userInputs") or []
    if inputs:                       # keep the last input the driver entered
        u = inputs[-1]
        for k in ["kWhRequested", "minutesAvailable", "requestedDeparture",
                  "milesRequested", "WhPerMile", "paymentRequired"]:
            row[k] = u.get(k)
    return row


# ---------- main ----------
start = pd.Timestamp(START, tz="UTC") if START else first_session_time().normalize()
print(f"Site: {SITE} | Year 1 starts: {start.date()}")

edges = [start + pd.DateOffset(months=i) for i in range(MONTHS + 1)]
chunks = []
for a, b in zip(edges[:-1], edges[1:]):
    f = OUT / f"{SITE}_{a:%Y-%m}.csv"
    if f.exists():                   # resume: skip months already downloaded
        print(f"{a:%Y-%m}: already downloaded")
        chunks.append(pd.read_csv(f))
        continue
    rows = fetch_range(a, b)
    df = pd.DataFrame([flatten(s) for s in rows])
    df.to_csv(f, index=False)
    print(f"{a:%Y-%m}: {len(df)} sessions")
    chunks.append(df)

df = pd.concat(chunks, ignore_index=True)

# Convert UTC strings to local time and add the duration columns
tz = df["timezone"].dropna().iloc[0]
for c in ["connectionTime", "disconnectTime", "doneChargingTime", "requestedDeparture"]:
    if c in df:
        df[c] = pd.to_datetime(df[c], utc=True).dt.tz_convert(tz)
df["plugged_hours"] = (df["disconnectTime"] - df["connectionTime"]).dt.total_seconds() / 3600
df["idle_hours"] = (df["disconnectTime"] - df["doneChargingTime"]).dt.total_seconds() / 3600

df.to_csv(OUT / f"{SITE}_year1.csv", index=False)
print(f"\nTotal sessions: {len(df)}")
print(f"With driver inputs (kWhRequested): {df['kWhRequested'].notna().mean():.0%}")
print(df[["kWhDelivered", "plugged_hours", "idle_hours"]].describe())

Site: jpl | Year 1 starts: 2021-09-13
2021-09: 51 sessions
2021-10: 0 sessions
2021-11: 0 sessions
2021-12: 0 sessions
2022-01: 0 sessions
2022-02: 0 sessions
2022-03: 0 sessions
2022-04: 0 sessions
2022-05: 0 sessions
2022-06: 0 sessions
2022-07: 0 sessions
2022-08: 0 sessions

Total sessions: 51
With driver inputs (kWhRequested): 96%
       kWhDelivered  plugged_hours  idle_hours
count     51.000000      51.000000   51.000000
mean      15.190922       6.441727    2.334700
std       12.773335       2.778755    2.608184
min        0.904000       0.385556   -0.016111
25%        6.513500       4.306944    0.009583
50%       12.123000       6.595833    1.628889
75%       18.466500       8.595694    3.589028
max       54.036000      11.925278    9.926389


In [11]:
df.describe(include="all")

,sessionID,siteID,clusterID,stationID,spaceID,userID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,timezone,kWhRequested,minutesAvailable,requestedDeparture,milesRequested,WhPerMile,paymentRequired,plugged_hours,idle_hours
count,51,51,51,51,51,49,51,51,51,51.000000,51,49.000000,49.000000,49,49.000000,49.000000,49,51.000000,51.000000
unique,51,1,1,41,41,46,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
top,1_1_193_820_2021-09-13 09:28:45.074062,0001,0001,1-1-178-817,AG-1F09,000012256,NaN,NaN,NaN,NaN,America/Los_Angeles,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN
freq,1,51,51,3,3,2,NaN,NaN,NaN,NaN,51,NaN,NaN,NaN,NaN,NaN,49,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,2021-09-13 09:03:29.647058-07:00,2021-09-13 15:29:59.862745-07:00,2021-09-13 13:09:54.941176-07:00,15.190922,NaN,34.183490,426.612245,2021-09-13 16:08:51.571428-07:00,119.714286,301.387755,NaN,6.441727,2.334700
min,NaN,NaN,NaN,NaN,NaN,NaN,2021-09-13 02:28:53-07:00,2021-09-13 06:30:03-07:00,2021-09-13 04:38:23-07:00,0.904000,NaN,5.600000,115.000000,2021-09-13 06:21:53-07:00,20.000000,237.000000,NaN,0.385556,-0.016111
25%,NaN,NaN,NaN,NaN,NaN,NaN,2021-09-13 07:24:31-07:00,2021-09-13 13:58:09.500000-07:00,2021-09-13 10:27:04-07:00,6.513500,NaN,14.040000,240.000000,2021-09-13 12:09:13-07:00,40.000000,274.000000,NaN,4.306944,0.009583
50%,NaN,NaN,NaN,NaN,NaN,NaN,2021-09-13 08:32:18-07:00,2021-09-13 15:39:40-07:00,2021-09-13 12:00:52-07:00,12.123000,NaN,25.800000,420.000000,2021-09-13 15:33:29-07:00,100.000000,284.000000,NaN,6.595833,1.628889
75%,NaN,NaN,NaN,NaN,NaN,NaN,2021-09-13 09:27:33.500000-07:00,2021-09-13 17:12:02.500000-07:00,2021-09-13 16:04:24.500000-07:00,18.466500,NaN,51.600000,540.000000,2021-09-13 18:32:18-07:00,175.000000,310.000000,NaN,8.595694,3.589028
max,NaN,NaN,NaN,NaN,NaN,NaN,2021-09-13 22:43:39-07:00,2021-09-14 07:46:28-07:00,2021-09-14 07:46:22-07:00,54.036000,NaN,100.014000,1467.000000,2021-09-14 10:15:39-07:00,422.000000,520.000000,NaN,11.925278,9.926389


In [12]:
df.head()

,sessionID,siteID,clusterID,stationID,spaceID,userID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,timezone,kWhRequested,minutesAvailable,requestedDeparture,milesRequested,WhPerMile,paymentRequired,plugged_hours,idle_hours
0,1_1_193_820_2021-09-13 09:28:45.074062,0001,0001,1-1-193-820,AG-1F04,000010817,2021-09-13 02:28:53-07:00,2021-09-13 06:30:03-07:00,2021-09-13 04:38:23-07:00,13.493,America/Los_Angeles,23.28,233.0,2021-09-13 06:21:53-07:00,80.0,291.0,True,4.019444,1.861111
1,1_1_179_799_2021-09-13 12:16:41.576939,0001,0001,1-1-179-799,AG-3F25,000000421,2021-09-13 05:16:42-07:00,2021-09-13 16:00:06-07:00,2021-09-13 08:00:01-07:00,17.263,America/Los_Angeles,43.75,504.0,2021-09-13 13:40:42-07:00,175.0,250.0,True,10.723333,8.001389
2,1_1_191_802_2021-09-13 12:28:07.955475,0001,0001,1-1-191-802,AG-4F44,000006453,2021-09-13 05:28:25-07:00,2021-09-13 17:23:56-07:00,2021-09-13 16:58:48-07:00,41.012,America/Los_Angeles,55.80,605.0,2021-09-13 15:33:25-07:00,200.0,279.0,True,11.925278,0.418889
3,1_1_193_829_2021-09-13 12:38:25.642441,0001,0001,1-1-193-829,AG-1F03,000000169,2021-09-13 05:38:33-07:00,2021-09-13 14:54:15-07:00,2021-09-13 11:19:04-07:00,18.618,America/Los_Angeles,25.80,480.0,2021-09-13 13:38:33-07:00,100.0,258.0,True,9.261667,3.586389
4,1_1_194_818_2021-09-13 12:45:44.299784,0001,0001,1-1-194-818,AG-1F14,000008102,2021-09-13 05:45:55-07:00,2021-09-13 14:12:49-07:00,2021-09-13 08:41:51-07:00,10.655,America/Los_Angeles,23.20,360.0,2021-09-13 11:45:55-07:00,80.0,290.0,True,8.448333,5.516111
